In [5]:
"""
================================================================
PURE PREMIUM PRICING MODEL
Negative Binomial (Frequency) × Gamma (Severity) = Pure Premium

Dataset : freMTPL2freq.csv  (PolicyID, ClaimNb, Exposure, Power,
                              CarAge, DriverAge, Brand, Gas,
                              Region, Density)
          freMTPL2sev.csv   (PolicyID, ClaimAmount)

Models:
  Frequency : Negative Binomial GLM  (handles overdispersion)
  Severity  : Gamma GLM              (handles right-skewed costs)
  Pure Premium = Predicted Frequency × Predicted Severity

HOW TO RUN:
  Place both CSVs in the same folder as this script, then run:
  python pure_premium_glm.py
================================================================
"""

'\n================================================================\nPURE PREMIUM PRICING MODEL\nNegative Binomial (Frequency) × Gamma (Severity) = Pure Premium\n\nDataset : freMTPL2freq.csv  (PolicyID, ClaimNb, Exposure, Power,\n                              CarAge, DriverAge, Brand, Gas,\n                              Region, Density)\n          freMTPL2sev.csv   (PolicyID, ClaimAmount)\n\nModels:\n  Frequency : Negative Binomial GLM  (handles overdispersion)\n  Severity  : Gamma GLM              (handles right-skewed costs)\n  Pure Premium = Predicted Frequency × Predicted Severity\n\nHOW TO RUN:\n  Place both CSVs in the same folder as this script, then run:\n  python pure_premium_glm.py\n================================================================\n'

In [3]:
!pip install matplotlib

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import LabelEncoder
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

In [7]:
#STEP 1: LOAD & MERGE DATA
print("Step 1:Loading data")
freq = pd.read_csv('freMTPLfreq.csv')
sev  = pd.read_csv('freMTPLsev.csv')

# Aggregate severity: one row per policy
sev_agg = (sev.groupby('PolicyID')['ClaimAmount']
              .sum()
              .reset_index()
              .rename(columns={'ClaimAmount': 'TotalClaimAmount'}))

df = freq.merge(sev_agg, on='PolicyID', how='left')
df['TotalClaimAmount'] = df['TotalClaimAmount'].fillna(0)

print(f"  Policies loaded : {len(df):,}")
print(f"  Total claims    : {df['ClaimNb'].sum():,}")


Step 1:Loading data
  Policies loaded : 413,169
  Total claims    : 16,181


In [9]:
#STEP 2: DATA CLEANING & PREPARATION
# Basic cleaning
df['Exposure']   = df['Exposure'].clip(0.001, 1.0)
df['ClaimNb']    = df['ClaimNb'].clip(lower=0)
df              = df[(df['DriverAge'] >= 18) & (df['DriverAge'] <= 100)]
df              = df[(df['CarAge']    >= 0)  & (df['CarAge']    <= 50)]
df.dropna(subset=['ClaimNb','Exposure','DriverAge','CarAge','Density'], inplace=True)
df              = df.reset_index(drop=True)

#Feature engineering
# Age bands  — U-shaped relationship with frequency
df['AgeBand'] = pd.cut(df['DriverAge'],
                        bins   = [17,25,35,50,65,110],
                        labels = ['Age_18_25','Age_26_35',
                                  'Age_36_50','Age_51_65','Age_65plus'])

# Car age bands
df['CarAgeBand'] = pd.cut(df['CarAge'],
                           bins   = [-1,2,5,10,15,200],
                           labels = ['Car_0_2','Car_3_5',
                                     'Car_6_10','Car_11_15','Car_15plus'])

# Log density — right-skewed so log-transform it
df['LogDensity'] = np.log1p(df['Density'])

# Clean categorical columns so statsmodels can read them
df['Gas']    = df['Gas'].astype(str).str.strip().str.capitalize()
df['Region'] = df['Region'].astype(str).str.strip()
df['Power']  = df['Power'].astype(str).str.strip()
df['Brand']  = df['Brand'].astype(str).str.strip()

# Log exposure — used as offset in frequency model
# The offset fixes its coefficient at 1.0, meaning:
#   log(E[ClaimNb]) = Xβ + log(Exposure)
# which is the same as:
#   E[ClaimNb] = Exposure × exp(Xβ)
df['LogExposure'] = np.log(df['Exposure'])

# Severity dataset — only policies with claims > 0 and amount > 0
df_sev = df[(df['ClaimNb'] > 0) & (df['TotalClaimAmount'] > 0)].copy()
df_sev['AvgSeverity'] = df_sev['TotalClaimAmount'] / df_sev['ClaimNb']

print(f"  Clean policies          : {len(df):,}")
print(f"  Policies with claims    : {len(df_sev):,}")
print(f"  Portfolio freq (exp-wtd): {df['ClaimNb'].sum()/df['Exposure'].sum():.4f}")

# Overdispersion check — if VMR >> 1, Negative Binomial beats Poisson
vmr = df['ClaimNb'].var() / df['ClaimNb'].mean()
print(f"  VMR (Var/Mean)          : {vmr:.3f}  → {'NB2 recommended' if vmr>1.05 else 'Poisson OK'}")


  Clean policies          : 413,087
  Policies with claims    : 15,387
  Portfolio freq (exp-wtd): 0.0698
  VMR (Var/Mean)          : 1.063  → NB2 recommended


In [ ]:
#STEP 3: NEGATIVE BINOMIAL REGRESSION FOR FREQUENCY MODEL
# Formula used:
#   ClaimNb ~ AgeBand + CarAgeBand + Gas + LogDensity + Power
#
# offset=LogExposure fixes the exposure coefficient at 1.0
# family=NegativeBinomial handles overdispersion via a dispersion
# parameter α where Var(Y) = μ + α·μ²
# link=log means:  log(μ) = Xβ  →  μ = exp(Xβ)

In [10]:
nb_formula = ('ClaimNb ~ AgeBand + CarAgeBand + Gas '
              '+ LogDensity + Power')

nb_model = smf.glm(
    formula = nb_formula,
    data    = df,
    family  = sm.families.NegativeBinomial(
                  link=sm.families.links.Log()),   # log link
    offset  = df['LogExposure']                    # exposure offset
).fit(disp=False)

print(nb_model.summary2().tables[1].round(4))
print(f"\n  AIC  : {nb_model.aic:.2f}")
print(f"  Obs  : {int(nb_model.nobs):,}")

# Add frequency predictions back to main dataframe
df['PredFreq'] = nb_model.predict(df)

                           Coef.  Std.Err.        z   P>|z|  [0.025  0.975]
Intercept                -2.5721    0.0469 -54.7938  0.0000 -2.6641 -2.4801
AgeBand[T.Age_26_35]     -0.7165    0.0318 -22.5069  0.0000 -0.7789 -0.6541
AgeBand[T.Age_36_50]     -0.7489    0.0296 -25.2885  0.0000 -0.8069 -0.6908
AgeBand[T.Age_51_65]     -0.8252    0.0312 -26.4733  0.0000 -0.8863 -0.7641
AgeBand[T.Age_65plus]    -0.8888    0.0365 -24.3261  0.0000 -0.9604 -0.8172
CarAgeBand[T.Car_3_5]     0.0405    0.0259   1.5635  0.1179 -0.0103  0.0913
CarAgeBand[T.Car_6_10]    0.1090    0.0238   4.5780  0.0000  0.0623  0.1557
CarAgeBand[T.Car_11_15]   0.0245    0.0255   0.9604  0.3369 -0.0255  0.0746
CarAgeBand[T.Car_15plus] -0.1631    0.0357  -4.5715  0.0000 -0.2330 -0.0932
Gas[T.Regular]           -0.1716    0.0177  -9.7240  0.0000 -0.2062 -0.1370
Power[T.e]                0.0938    0.0284   3.3017  0.0010  0.0381  0.1495
Power[T.f]                0.0902    0.0280   3.2194  0.0013  0.0353  0.1452
Power[T.g]  

In [11]:
#STEP 4: GAMMA GLM FOR SEVERITY MODEL
# Formula used:
#   AvgSeverity ~ AgeBand + CarAgeBand + Gas + LogDensity + Power
#
# family=Gamma  — always positive, right-skewed  ✓
# link=log      — multiplicative rating factors  ✓
# No offset needed for severity (it's per-claim, not per-year)

In [12]:
gamma_formula = ('AvgSeverity ~ AgeBand + CarAgeBand + Gas '
                 '+ LogDensity + Power')

gamma_model = smf.glm(
    formula = gamma_formula,
    data    = df_sev,
    family  = sm.families.Gamma(
                  link=sm.families.links.Log())    # log link
).fit(disp=False)

print(gamma_model.summary2().tables[1].round(4))
print(f"\n  AIC  : {gamma_model.aic:.2f}")
print(f"  Obs  : {int(gamma_model.nobs):,}")

# Add severity predictions — predict for ALL policies
# (every policy has an expected severity in the event of a claim)
df['PredSev'] = gamma_model.predict(df)

                           Coef.  Std.Err.        z   P>|z|  [0.025  0.975]
Intercept                 8.3204    0.2268  36.6902  0.0000  7.8759  8.7648
AgeBand[T.Age_26_35]     -0.5401    0.1536  -3.5156  0.0004 -0.8412 -0.2390
AgeBand[T.Age_36_50]     -0.6776    0.1423  -4.7606  0.0000 -0.9566 -0.3986
AgeBand[T.Age_51_65]     -0.6494    0.1502  -4.3250  0.0000 -0.9437 -0.3551
AgeBand[T.Age_65plus]    -0.4744    0.1764  -2.6894  0.0072 -0.8202 -0.1287
CarAgeBand[T.Car_3_5]     0.0418    0.1249   0.3349  0.7377 -0.2030  0.2867
CarAgeBand[T.Car_6_10]   -0.0496    0.1145  -0.4334  0.6647 -0.2740  0.1748
CarAgeBand[T.Car_11_15]   0.0344    0.1227   0.2805  0.7791 -0.2060  0.2749
CarAgeBand[T.Car_15plus] -0.1064    0.1713  -0.6212  0.5345 -0.4421  0.2293
Gas[T.Regular]            0.0392    0.0851   0.4607  0.6450 -0.1276  0.2060
Power[T.e]               -0.0076    0.1376  -0.0554  0.9558 -0.2773  0.2620
Power[T.f]                0.2007    0.1355   1.4819  0.1384 -0.0648  0.4662
Power[T.g]  

In [13]:
#STEP 5: CALCULATING PURE PREMIUM
df['PurePremium'] = df['PredFreq'] * df['PredSev']

# For annualised pure premium, divide by exposure
# (PredFreq already accounts for exposure via offset)
df['PurePremium_Annual'] = (df['PredFreq'] / df['Exposure']) * df['PredSev']

print(f"  Mean Pure Premium (annual) : €{df['PurePremium_Annual'].mean():,.2f}")
print(f"  Median                     : €{df['PurePremium_Annual'].median():,.2f}")
print(f"  Min                        : €{df['PurePremium_Annual'].min():,.2f}")
print(f"  Max (99th pct)             : €{df['PurePremium_Annual'].quantile(0.99):,.2f}")

  Mean Pure Premium (annual) : €1,367.82
  Median                     : €254.58
  Min                        : €52.96
  Max (99th pct)             : €20,104.43


In [14]:
#STEP 6: SAMPLE POLICYHOLDER PRICING
# Build a small table of 4 example policyholders
# Using the modal (most common) category as base
base_power  = df['Power'].mode()[0]
base_region = df['Region'].mode()[0]
base_brand  = df['Brand'].mode()[0]

samples = pd.DataFrame({
    'PolicyID'   : [99001, 99002, 99003, 99004],
    'DriverAge'  : [22,    42,    58,    70   ],
    'CarAge'     : [1,     7,     4,     12   ],
    'Gas'        : ['Regular','Diesel','Diesel','Regular'],
    'Density'    : [3000,  500,   1200,  200  ],
    'Exposure'   : [1.0,   1.0,   1.0,   1.0 ],
    'Power'      : [base_power]*4,
    'Region'     : [base_region]*4,
    'Brand'      : [base_brand]*4,
    'ClaimNb'    : [0,0,0,0],           # needed for formula parsing
    'TotalClaimAmount': [0,0,0,0],
})

# Apply same feature engineering as training data
samples['AgeBand'] = pd.cut(samples['DriverAge'],
                             bins=[17,25,35,50,65,110],
                             labels=['Age_18_25','Age_26_35',
                                     'Age_36_50','Age_51_65','Age_65plus'])
samples['CarAgeBand'] = pd.cut(samples['CarAge'],
                                bins=[-1,2,5,10,15,200],
                                labels=['Car_0_2','Car_3_5',
                                        'Car_6_10','Car_11_15','Car_15plus'])
samples['LogDensity']  = np.log1p(samples['Density'])
samples['LogExposure'] = np.log(samples['Exposure'])
samples['AvgSeverity'] = 0   # placeholder for gamma prediction

samples['PredFreq']    = nb_model.predict(samples)
samples['PredSev']     = gamma_model.predict(samples)
samples['PurePremium'] = samples['PredFreq'] * samples['PredSev']
samples['LoadedPrem']  = samples['PurePremium'] * 1.30   # 30% loading

labels = ['Young (22, new car)', 'Middle-aged (42)',
          'Experienced (58)',    'Elderly (70, old car)']
print(f"\n  {'Profile':<25} {'Freq':>8} {'Severity':>10} "
      f"{'Pure Prem':>11} {'Loaded (30%)':>13}")
print("  " + "-" * 72)
for i, row in samples.iterrows():
    print(f"  {labels[i]:<25} {row['PredFreq']:>8.4f} "
          f"€{row['PredSev']:>9,.0f} "
          f"€{row['PurePremium']:>10,.0f} "
          f"€{row['LoadedPrem']:>12,.0f}")


  Profile                       Freq   Severity   Pure Prem  Loaded (30%)
  ------------------------------------------------------------------------
  Young (22, new car)         0.1623 €    3,504 €       569 €         740
  Middle-aged (42)            0.0843 €    1,780 €       150 €         195
  Experienced (58)            0.0799 €    1,921 €       154 €         200
  Elderly (70, old car)       0.0516 €    2,582 €       133 €         173


In [18]:
!pip install seaborn
import seaborn as sns

In [19]:
#STEP 7:VISUALIZATIONS
print("\nBuilding plots...")

C_BLUE   = '#1B4F72'
C_TEAL   = '#117A65'
C_ORANGE = '#D35400'
C_RED    = '#A93226'
C_GREY   = '#5D6D7E'
C_PURPLE = '#6C3483'
BG       = '#FAFAFA'

plt.rcParams.update({
    'figure.facecolor' : BG,  'axes.facecolor'  : BG,
    'axes.edgecolor'   : '#CCCCCC',
    'axes.grid'        : True, 'grid.alpha'      : 0.35,
    'grid.linestyle'   : '--', 'font.size'       : 10,
    'axes.titlesize'   : 11,   'axes.labelsize'  : 10,
    'legend.fontsize'  : 9,
})

fig = plt.figure(figsize=(18, 24))
fig.patch.set_facecolor(BG)
fig.suptitle(
    'Pure Premium GLM — Negative Binomial × Gamma\n'
    'freMTPL2 French Motor Insurance Dataset',
    fontsize=17, fontweight='bold', y=0.995, color='#1C2833'
)
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.50, wspace=0.38)

portfolio_freq = df['ClaimNb'].sum() / df['Exposure'].sum()


# ── Chart 1: Actual vs Predicted Frequency ───────────────────
ax1 = fig.add_subplot(gs[0, 0])
# Bin into age groups and compare
age_compare = df.groupby('AgeBand', observed=True).agg(
    ActualFreq = ('ClaimNb',   lambda x: x.sum() / df.loc[x.index,'Exposure'].sum()),
    PredFreq   = ('PredFreq',  lambda x: x.sum() / df.loc[x.index,'Exposure'].sum()),
).reset_index()
x = np.arange(len(age_compare))
ax1.bar(x - 0.2, age_compare['ActualFreq'], 0.38,
        color=C_BLUE,   label='Actual',    edgecolor='white', alpha=0.9)
ax1.bar(x + 0.2, age_compare['PredFreq'],  0.38,
        color=C_ORANGE, label='Predicted', edgecolor='white', alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(age_compare['AgeBand'].astype(str), fontsize=9)
ax1.set_xlabel('Driver Age Band')
ax1.set_ylabel('Claim Frequency')
ax1.set_title('NB2 Model: Actual vs Predicted\nFrequency by Age Band',
              fontweight='bold')
ax1.legend()


# ── Chart 2: Actual vs Predicted Severity ────────────────────
ax2 = fig.add_subplot(gs[0, 1])
sev_compare = df_sev.groupby('AgeBand', observed=True).agg(
    ActualSev = ('AvgSeverity', 'mean'),
    PredSev   = ('AgeBand',
                 lambda x: gamma_model.predict(df_sev.loc[x.index]).mean()),
).reset_index()
x2 = np.arange(len(sev_compare))
ax2.bar(x2 - 0.2, sev_compare['ActualSev'], 0.38,
        color=C_TEAL,   label='Actual',    edgecolor='white', alpha=0.9)
ax2.bar(x2 + 0.2, sev_compare['PredSev'],  0.38,
        color=C_PURPLE, label='Predicted', edgecolor='white', alpha=0.85)
ax2.set_xticks(x2)
ax2.set_xticklabels(sev_compare['AgeBand'].astype(str), fontsize=9)
ax2.set_xlabel('Driver Age Band')
ax2.set_ylabel('Avg Claim Severity (€)')
ax2.set_title('Gamma Model: Actual vs Predicted\nSeverity by Age Band',
              fontweight='bold')
ax2.legend()


# ── Chart 3: Pure Premium by Age Band ────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
pp_age = df.groupby('AgeBand', observed=True)['PurePremium_Annual'].mean().reset_index()
colors3 = [C_RED, C_ORANGE, C_TEAL, C_BLUE, C_GREY]
bars3   = ax3.bar(pp_age['AgeBand'].astype(str),
                  pp_age['PurePremium_Annual'],
                  color=colors3, edgecolor='white', alpha=0.9)
for bar, v in zip(bars3, pp_age['PurePremium_Annual']):
    ax3.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 2,
             f'€{v:.0f}', ha='center', fontsize=9, fontweight='bold')
ax3.set_xlabel('Driver Age Band')
ax3.set_ylabel('Mean Pure Premium (€/yr)')
ax3.set_title('Pure Premium by Driver Age Band\n(Frequency × Severity)',
              fontweight='bold')


# ── Chart 4: NB2 Coefficient Plot ────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
nb_coef = nb_model.params.drop('Intercept')
nb_ci   = nb_model.conf_int().drop('Intercept')
colors4 = [C_RED if v > 0 else C_TEAL for v in nb_coef.values]
y_pos   = np.arange(len(nb_coef))

ax4.barh(y_pos, nb_coef.values,
         color=colors4, edgecolor='white', alpha=0.85)
ax4.errorbar(nb_coef.values, y_pos,
             xerr=[nb_coef.values - nb_ci[0].values,
                   nb_ci[1].values - nb_coef.values],
             fmt='none', color='black', capsize=3, lw=1.2)
ax4.axvline(0, color='black', lw=1.5, ls='--')
ax4.set_yticks(y_pos)
ax4.set_yticklabels([n[:22] for n in nb_coef.index], fontsize=8)
ax4.set_xlabel('Coefficient (log scale)  →  exp(coef) = rate ratio')
ax4.set_title('NB2 Frequency Model\nCoefficients with 95% CI',
              fontweight='bold')
ax4.text(0.98, 0.02, 'Red = increases freq\nTeal = decreases freq',
         transform=ax4.transAxes, ha='right', fontsize=8,
         bbox=dict(boxstyle='round', fc='white', ec=C_GREY, alpha=0.8))


# ── Chart 5: Gamma Coefficient Plot ──────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
g_coef = gamma_model.params.drop('Intercept')
g_ci   = gamma_model.conf_int().drop('Intercept')
colors5 = [C_RED if v > 0 else C_TEAL for v in g_coef.values]
y_pos5  = np.arange(len(g_coef))

ax5.barh(y_pos5, g_coef.values,
         color=colors5, edgecolor='white', alpha=0.85)
ax5.errorbar(g_coef.values, y_pos5,
             xerr=[g_coef.values - g_ci[0].values,
                   g_ci[1].values - g_coef.values],
             fmt='none', color='black', capsize=3, lw=1.2)
ax5.axvline(0, color='black', lw=1.5, ls='--')
ax5.set_yticks(y_pos5)
ax5.set_yticklabels([n[:22] for n in g_coef.index], fontsize=8)
ax5.set_xlabel('Coefficient (log scale)  →  exp(coef) = severity multiplier')
ax5.set_title('Gamma Severity Model\nCoefficients with 95% CI',
              fontweight='bold')


# ── Chart 6: Pure Premium Distribution ───────────────────────
ax6 = fig.add_subplot(gs[1, 2])
pp_plot = df['PurePremium_Annual'].clip(
    upper=df['PurePremium_Annual'].quantile(0.99)
)
ax6.hist(pp_plot, bins=60, color=C_BLUE, edgecolor='white', alpha=0.85)
ax6.axvline(pp_plot.mean(),   color=C_RED,    ls='--', lw=2,
            label=f"Mean = €{pp_plot.mean():.0f}")
ax6.axvline(pp_plot.median(), color=C_ORANGE, ls='--', lw=2,
            label=f"Median = €{pp_plot.median():.0f}")
ax6.set_xlabel('Annual Pure Premium (€)')
ax6.set_ylabel('Policy Count')
ax6.set_title('Pure Premium Distribution\n(clipped at 99th percentile)',
              fontweight='bold')
ax6.legend()


# ── Chart 7: Frequency residuals ─────────────────────────────
ax7 = fig.add_subplot(gs[2, 0])
nb_resid = nb_model.resid_pearson
ax7.scatter(nb_model.fittedvalues, nb_resid,
            alpha=0.15, s=5, color=C_BLUE)
ax7.axhline(0,  color=C_RED,  ls='--', lw=1.5)
ax7.axhline(2,  color=C_GREY, ls=':',  lw=1)
ax7.axhline(-2, color=C_GREY, ls=':',  lw=1)
ax7.set_xlabel('Fitted Values (log scale)')
ax7.set_ylabel('Pearson Residual')
ax7.set_title('NB2 Residuals vs Fitted\n(should be centred around 0)',
              fontweight='bold')
ax7.set_ylim(-6, 6)


# ── Chart 8: Severity residuals ───────────────────────────────
ax8 = fig.add_subplot(gs[2, 1])
g_resid = gamma_model.resid_pearson
ax8.scatter(gamma_model.fittedvalues, g_resid,
            alpha=0.2, s=5, color=C_TEAL)
ax8.axhline(0,  color=C_RED,  ls='--', lw=1.5)
ax8.axhline(2,  color=C_GREY, ls=':',  lw=1)
ax8.axhline(-2, color=C_GREY, ls=':',  lw=1)
ax8.set_xlabel('Fitted Values (log scale)')
ax8.set_ylabel('Pearson Residual')
ax8.set_title('Gamma Residuals vs Fitted\n(should be centred around 0)',
              fontweight='bold')
ax8.set_ylim(-6, 6)


# ── Chart 9: Pure Premium by Gas type ────────────────────────
ax9 = fig.add_subplot(gs[2, 2])
pp_gas = df.groupby('Gas')['PurePremium_Annual'].mean().reset_index()
bars9  = ax9.bar(pp_gas['Gas'], pp_gas['PurePremium_Annual'],
                 color=[C_BLUE, C_ORANGE], edgecolor='white',
                 alpha=0.9, width=0.4)
for bar, v in zip(bars9, pp_gas['PurePremium_Annual']):
    ax9.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 1,
             f'€{v:.0f}', ha='center', fontsize=11, fontweight='bold')
ax9.set_xlabel('Fuel Type')
ax9.set_ylabel('Mean Annual Pure Premium (€)')
ax9.set_title('Pure Premium by Fuel Type',
              fontweight='bold')


# ── Chart 10: Pure Premium heatmap (Age × Car Age) ───────────
ax10 = fig.add_subplot(gs[3, :2])
import seaborn as sns
pivot = df.groupby(['AgeBand','CarAgeBand'], observed=True)['PurePremium_Annual'].mean().unstack()
sns.heatmap(pivot, ax=ax10,
            cmap='YlOrRd', annot=True, fmt='.0f',
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Pure Premium (€/yr)'})
ax10.set_title('Pure Premium Heatmap — Driver Age × Car Age\n'
               '(GLM predicted:  NB2 Frequency  ×  Gamma Severity)',
               fontweight='bold')
ax10.set_xlabel('Car Age Band')
ax10.set_ylabel('Driver Age Band')


# ── Chart 11: Sample policyholder premiums ───────────────────
ax11 = fig.add_subplot(gs[3, 2])
ax11.axis('off')

table_data = [
    ['Profile',         'Freq',   'Severity', 'Pure Prem', 'Loaded'],
    ['─'*14,            '─'*6,    '─'*9,      '─'*9,       '─'*7  ],
]
for i, row in samples.iterrows():
    table_data.append([
        labels[i][:14],
        f"{row['PredFreq']:.4f}",
        f"€{row['PredSev']:,.0f}",
        f"€{row['PurePremium']:,.0f}",
        f"€{row['LoadedPrem']:,.0f}",
    ])

y_start = 0.95
ax11.text(0.5, 1.00, 'Sample Policy Quotes', transform=ax11.transAxes,
          ha='center', va='top', fontsize=11, fontweight='bold', color=C_BLUE)

col_x = [0.00, 0.32, 0.50, 0.68, 0.86]
for row_i, row in enumerate(table_data):
    y = y_start - row_i * 0.13
    for col_i, cell in enumerate(row):
        weight = 'bold' if row_i == 0 else 'normal'
        color  = C_BLUE if row_i == 0 else '#1C2833'
        ax11.text(col_x[col_i], y, cell,
                  transform=ax11.transAxes, fontsize=8.5,
                  va='top', fontweight=weight, color=color)

ax11.text(0.5, -0.02,
          'Loaded Premium = Pure Premium × 1.30\n(30% for expenses + profit)',
          transform=ax11.transAxes, ha='center', fontsize=8,
          color=C_GREY, style='italic')

# ── Save ─────────────────────────────────────────────────────
plt.savefig('pure_premium_glm.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.close()
print("  ✓ Chart saved → pure_premium_glm.png")


Building plots...
  ✓ Chart saved → pure_premium_glm.png


In [20]:
#FINAL SUMMARY
print("\n" + "=" * 55)
print("  MODEL SUMMARY")
print("=" * 55)
print(f"  NB2  Frequency  AIC : {nb_model.aic:>12.2f}")
print(f"  Gamma Severity  AIC : {gamma_model.aic:>12.2f}")
print(f"\n  Portfolio mean pure premium : €{df['PurePremium_Annual'].mean():>8,.2f}/yr")
print(f"  With 30% loading            : €{df['PurePremium_Annual'].mean()*1.30:>8,.2f}/yr")
print("\n  DONE")



  MODEL SUMMARY
  NB2  Frequency  AIC :    134881.82
  Gamma Severity  AIC :    312869.29

  Portfolio mean pure premium : €1,367.82/yr
  With 30% loading            : €1,778.17/yr

  DONE
